# AlpCAN — CXR Pipeline: TorchXRayVision Baseline

**Amaç:** NIH ChestX-ray14 tam veri seti (112,120 görüntü, 30,805 hasta) üzerinde TorchXRayVision DenseNet-121 modelinin performansını değerlendirmek. AlpCAN CXR Pipeline Agent 7B baseline.

**İçerik:**
1. Veri seti keşfi — NIH ChestX-ray14 tam istatistikler
2. TorchXRayVision pre-trained model yükleme
3. Tam dataset inference (112K+ görüntü)
4. AUC-ROC performans değerlendirmesi
5. Threshold analizi (Nodule/Mass odaklı)
6. Ensemble simülasyonu
7. BBox ile lokalizasyon doğrulama

**Dataset:** `nih-chest-xrays/data` (45GB, orijinal çözünürlük)

**Referans:** Wang et al., "ChestX-ray8: Hospital-scale Chest X-ray Database and Benchmarks" CVPR 2017

**GPU:** Kaggle T4 16GB

In [ ]:
# Kütüphaneleri kur
!pip install -q torchxrayvision scikit-learn

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

import torch
import torchxrayvision as xrv
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score

print(f"PyTorch: {torch.__version__}")
print(f"TorchXRayVision: {xrv.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"CUDA: {torch.cuda.is_available()}")

---
## 1. Veri Seti Keşfi — NIH ChestX-ray14

In [ ]:
# Dataset yolunu bul
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

# Data_Entry CSV
csv_candidates = list(INPUT_DIR.rglob("Data_Entry*.csv"))
if not csv_candidates:
    raise FileNotFoundError("Data_Entry CSV bulunamadı! Dataset eklenmiş mi?")
LABELS_CSV = csv_candidates[0]

# Görüntü klasörleri (images_001 - images_012)
all_img_dirs = sorted(INPUT_DIR.rglob("images"))
# images/ alt klasörlerini bul (images_001/images/, images_002/images/ ...)
IMG_DIRS = [d for d in all_img_dirs if d.is_dir() and list(d.glob('*.png'))]

# Tek images klasörü olabilir (sample dataset) veya birden fazla
if not IMG_DIRS:
    # Doğrudan PNG'leri ara
    all_pngs = list(INPUT_DIR.rglob("*.png"))
    if all_pngs:
        IMG_DIRS = list(set(p.parent for p in all_pngs[:10]))

print(f"Labels CSV: {LABELS_CSV}")
print(f"Image directories: {len(IMG_DIRS)}")
for d in IMG_DIRS[:5]:
    n_files = len(list(d.glob('*.png')))
    print(f"  {d}: {n_files:,} PNG")

In [ ]:
# Tüm görüntü yollarını index'le
image_index = {}
for img_dir in IMG_DIRS:
    for png in img_dir.glob('*.png'):
        image_index[png.name] = png

print(f"Toplam indekslenen görüntü: {len(image_index):,}")

In [ ]:
# Labels CSV'yi oku
df = pd.read_csv(LABELS_CSV)
print(f"CSV kayıt sayısı: {len(df):,}")
print(f"Hasta sayısı: {df['Patient ID'].nunique():,}")
print(f"\nSütunlar: {list(df.columns)}")
print(f"\nÖrnek etiketler:")
print(df['Finding Labels'].value_counts().head(10))
df.head()

In [ ]:
# 14 patoloji etiketlerini binary sütunlara dönüştür
ALL_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Emphysema', 'Fibrosis', 'Hernia',
    'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening',
    'Pneumonia', 'Pneumothorax'
]

for label in ALL_LABELS:
    df[label] = df['Finding Labels'].apply(lambda x: 1 if label in str(x) else 0)

df['No Finding'] = df['Finding Labels'].apply(lambda x: 1 if x == 'No Finding' else 0)

# Tam dağılım
print("=" * 65)
print("NIH ChestX-ray14 — Patoloji Dağılımı (TAM DATASET)")
print("=" * 65)

label_counts = df[ALL_LABELS].sum().sort_values(ascending=False)
for label, count in label_counts.items():
    pct = count / len(df) * 100
    marker = " ★ AlpCAN" if label in ('Nodule', 'Mass') else ""
    print(f"  {label:>20s}: {count:>6,} ({pct:>5.1f}%){marker}")

no_finding = df['No Finding'].sum()
print(f"  {'No Finding':>20s}: {no_finding:>6,} ({no_finding/len(df)*100:>5.1f}%)")
print(f"\n  Toplam görüntü: {len(df):,}")
print(f"  Toplam hasta:   {df['Patient ID'].nunique():,}")
print(f"  Multi-label:    {(df[ALL_LABELS].sum(axis=1) > 1).sum():,} görüntü birden fazla patoloji içeriyor")

In [ ]:
# Görselleştirme: Patoloji dağılımı + yaş/cinsiyet analizi
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Patoloji dağılımı
colors = ['#e74c3c' if l in ('Nodule', 'Mass') else '#3498db' for l in label_counts.index]
label_counts.plot(kind='barh', ax=axes[0, 0], color=colors)
axes[0, 0].set_title('Patoloji Dağılımı (N=112,120)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Görüntü Sayısı')
axes[0, 0].invert_yaxis()

# 2. Hasta başına bulgu sayısı
finding_per_image = df[ALL_LABELS].sum(axis=1)
finding_dist = finding_per_image.value_counts().sort_index()
axes[0, 1].bar(finding_dist.index, finding_dist.values, color='#2c3e50')
axes[0, 1].set_title('Görüntü Başına Bulgu Sayısı', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Bulgu Sayısı')
axes[0, 1].set_ylabel('Görüntü Sayısı')

# 3. Yaş dağılımı
if 'Patient Age' in df.columns:
    age_data = df['Patient Age'].dropna()
    age_data = age_data[age_data < 120]  # Outlier filtrele
    axes[1, 0].hist(age_data, bins=50, color='#2ecc71', edgecolor='white')
    axes[1, 0].set_title(f'Yaş Dağılımı (median={age_data.median():.0f})', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Yaş')
    axes[1, 0].set_ylabel('Sayı')

# 4. Cinsiyet dağılımı + patoloji
if 'Patient Gender' in df.columns:
    gender_pathology = df.groupby('Patient Gender')[ALL_LABELS].mean()
    gender_pathology[['Nodule', 'Mass', 'Infiltration', 'Effusion', 'Atelectasis']].T.plot(
        kind='bar', ax=axes[1, 1], color=['#3498db', '#e74c3c']
    )
    axes[1, 1].set_title('Cinsiyete Göre Patoloji Oranları', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Oran')
    axes[1, 1].legend(title='Cinsiyet')
    axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_dataset_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: cxr_dataset_overview.png")

In [ ]:
# BBox verisi — lokalize edilmiş bulgular
bbox_candidates = list(INPUT_DIR.rglob("BBox_List*.csv"))
if bbox_candidates:
    bbox_df = pd.read_csv(bbox_candidates[0])
    print(f"BBox kayıt sayısı: {len(bbox_df):,}")
    print(f"BBox sütunları: {list(bbox_df.columns)}")
    print(f"\nBBox patoloji dağılımı:")
    print(bbox_df['Finding Label'].value_counts())
    bbox_df.head()
else:
    print("BBox verisi bulunamadı")
    bbox_df = None

---
## 2. TorchXRayVision Model Yükleme

In [ ]:
# TorchXRayVision DenseNet-121 — tüm public CXR dataset'lerinden eğitilmiş
model = xrv.models.DenseNet(weights="densenet121-res224-all")
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model: DenseNet-121 (densenet121-res224-all)")
print(f"Device: {device}")
print(f"\nModel çıkış etiketleri ({len(model.pathologies)}):")
for i, p in enumerate(model.pathologies):
    print(f"  [{i:>2}] {p}")

# Model parametreleri
n_params = sum(p.numel() for p in model.parameters())
print(f"\nParametre sayısı: {n_params:,} ({n_params/1e6:.1f}M)")

In [ ]:
# NIH ↔ TorchXRayVision etiket eşleştirmesi
TXRV_LABELS = list(model.pathologies)

LABEL_MAP = {}
for nih_label in ALL_LABELS:
    for txrv_label in TXRV_LABELS:
        if nih_label.lower().replace('_', '') == txrv_label.lower().replace('_', ''):
            LABEL_MAP[nih_label] = txrv_label
            break

print("NIH → TorchXRayVision Etiket Eşleştirmesi:")
print("-" * 50)
for nih, txrv in sorted(LABEL_MAP.items()):
    txrv_idx = TXRV_LABELS.index(txrv)
    print(f"  {nih:>20s} → [{txrv_idx:>2}] {txrv}")

unmatched = set(ALL_LABELS) - set(LABEL_MAP.keys())
print(f"\n  Eşleşen: {len(LABEL_MAP)}/{len(ALL_LABELS)}")
if unmatched:
    print(f"  Eşleşmeyen: {unmatched}")

---
## 3. Tam Dataset Inference

In [ ]:
def load_and_preprocess(image_path):
    """CXR görüntüsünü TorchXRayVision formatına çevir.
    
    - Grayscale oku
    - 224x224 resize
    - [0,255] → [-1024, 1024] normalize (TorchXRayVision convention)
    """
    img = Image.open(image_path).convert('L')
    img = img.resize((224, 224), Image.LANCZOS)
    img = np.array(img, dtype=np.float32)
    
    # TorchXRayVision normalizasyonu
    img = xrv.datasets.normalize(img, 255)
    
    # (H,W) → (1,H,W)
    img = img[np.newaxis, :, :]
    return torch.from_numpy(img)


# Hızlı test
test_name = list(image_index.keys())[0]
test_img = load_and_preprocess(image_index[test_name])
print(f"Test: {test_name}")
print(f"Shape: {test_img.shape}, Range: [{test_img.min():.1f}, {test_img.max():.1f}]")

with torch.no_grad():
    out = model(test_img.unsqueeze(0).to(device)).cpu().numpy()[0]
print(f"Output shape: {out.shape}")
for i, (label, score) in enumerate(zip(model.pathologies, out)):
    if score > 0.3:
        print(f"  [{i}] {label}: {score:.4f} ⚠️")

In [ ]:
# Tam dataset inference
# 112K görüntü — batch processing ile GPU'yu verimli kullan
BATCH_SIZE = 64

# Sadece mevcut görüntüleri filtrele
df['exists'] = df['Image Index'].apply(lambda x: x in image_index)
eval_df = df[df['exists']].reset_index(drop=True)
print(f"Mevcut görüntü: {len(eval_df):,} / {len(df):,}")

all_predictions = []
all_image_names = []
skipped = 0
start_time = time.time()

n_batches = (len(eval_df) + BATCH_SIZE - 1) // BATCH_SIZE
print(f"\nInference başlıyor: {len(eval_df):,} görüntü, {n_batches:,} batch (batch_size={BATCH_SIZE})")
print("-" * 60)

for batch_start in range(0, len(eval_df), BATCH_SIZE):
    batch_df = eval_df.iloc[batch_start:batch_start + BATCH_SIZE]
    batch_tensors = []
    batch_names = []

    for _, row in batch_df.iterrows():
        img_name = row['Image Index']
        img_path = image_index.get(img_name)
        if img_path is None:
            skipped += 1
            continue
        try:
            tensor = load_and_preprocess(img_path)
            batch_tensors.append(tensor)
            batch_names.append(img_name)
        except Exception:
            skipped += 1
            continue

    if not batch_tensors:
        continue

    batch_input = torch.stack(batch_tensors).to(device)
    with torch.no_grad():
        batch_output = model(batch_input)
        batch_preds = batch_output.cpu().numpy()

    all_predictions.append(batch_preds)
    all_image_names.extend(batch_names)

    processed = len(all_image_names)
    if processed % 5000 < BATCH_SIZE or batch_start + BATCH_SIZE >= len(eval_df):
        elapsed = time.time() - start_time
        speed = processed / elapsed
        eta = (len(eval_df) - processed) / speed if speed > 0 else 0
        print(f"  {processed:>7,}/{len(eval_df):,} ({processed/len(eval_df)*100:>5.1f}%) "
              f"| {speed:.0f} img/s | ETA: {eta/60:.1f} min")

# Birleştir
predictions = np.concatenate(all_predictions, axis=0)
elapsed = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"✅ Inference tamamlandı!")
print(f"   İşlenen: {predictions.shape[0]:,} görüntü")
print(f"   Atlanan: {skipped}")
print(f"   Süre: {elapsed/60:.1f} dakika ({predictions.shape[0]/elapsed:.0f} img/s)")
print(f"   Prediction shape: {predictions.shape}")

---
## 4. Performans Değerlendirmesi — AUC-ROC

In [ ]:
# Ground truth al
eval_processed = eval_df[eval_df['Image Index'].isin(all_image_names)].reset_index(drop=True)
assert len(eval_processed) == len(predictions), "Boyut uyuşmazlığı!"

# AUC-ROC + AP (Average Precision) hesapla
print("=" * 70)
print("TorchXRayVision DenseNet-121 — NIH ChestX-ray14 Performansı")
print(f"N = {len(predictions):,} görüntü")
print("=" * 70)
print(f"{'Patoloji':>20s} | {'AUC-ROC':>8s} | {'AP':>8s} | {'N_pos':>7s} | {'Prevalans':>9s}")
print("-" * 70)

auc_results = {}
ap_results = {}

for nih_label, txrv_label in LABEL_MAP.items():
    txrv_idx = TXRV_LABELS.index(txrv_label)
    y_true = eval_processed[nih_label].values
    y_score = predictions[:, txrv_idx]

    n_pos = y_true.sum()
    prevalence = n_pos / len(y_true) * 100

    if n_pos > 0 and n_pos < len(y_true):
        auc = roc_auc_score(y_true, y_score)
        ap = average_precision_score(y_true, y_score)
        auc_results[nih_label] = auc
        ap_results[nih_label] = ap
        marker = " ★" if nih_label in ('Nodule', 'Mass') else ""
        print(f"  {nih_label:>20s} | {auc:>8.4f} | {ap:>8.4f} | {n_pos:>7,} | {prevalence:>8.1f}%{marker}")
    else:
        print(f"  {nih_label:>20s} | {'N/A':>8s} | {'N/A':>8s} | {n_pos:>7,} | {prevalence:>8.1f}%")

print("-" * 70)
if auc_results:
    mean_auc = np.mean(list(auc_results.values()))
    mean_ap = np.mean(list(ap_results.values()))
    print(f"  {'Ortalama':>20s} | {mean_auc:>8.4f} | {mean_ap:>8.4f} |")

    cancer_labels = [l for l in ('Nodule', 'Mass') if l in auc_results]
    if cancer_labels:
        cancer_auc = np.mean([auc_results[l] for l in cancer_labels])
        cancer_ap = np.mean([ap_results[l] for l in cancer_labels])
        print(f"  {'Nodule+Mass':>20s} | {cancer_auc:>8.4f} | {cancer_ap:>8.4f} | ← AlpCAN odak")

In [ ]:
# Wang et al. 2017 orijinal sonuçları ile karşılaştırma
WANG_AUC = {
    'Atelectasis': 0.716, 'Cardiomegaly': 0.814, 'Effusion': 0.784,
    'Infiltration': 0.609, 'Mass': 0.706, 'Nodule': 0.671,
    'Pneumonia': 0.633, 'Pneumothorax': 0.806, 'Consolidation': 0.708,
    'Edema': 0.805, 'Emphysema': 0.833, 'Fibrosis': 0.786,
    'Pleural_Thickening': 0.684, 'Hernia': 0.872
}

print("\n" + "=" * 60)
print("TorchXRayVision vs Wang et al. 2017 (CVPR)")
print("=" * 60)
print(f"{'Patoloji':>20s} | {'Bizim':>8s} | {'Wang':>8s} | {'Fark':>8s}")
print("-" * 55)

improvements = []
for label in sorted(auc_results.keys()):
    ours = auc_results[label]
    wang = WANG_AUC.get(label, 0)
    diff = ours - wang
    improvements.append(diff)
    arrow = "↑" if diff > 0 else "↓"
    print(f"  {label:>20s} | {ours:>8.4f} | {wang:>8.4f} | {diff:>+7.4f} {arrow}")

print("-" * 55)
print(f"  {'Ortalama fark':>20s} | {'':>8s} | {'':>8s} | {np.mean(improvements):>+7.4f}")

In [ ]:
# AUC-ROC ve ROC eğrisi görselleştirmesi
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# 1. AUC karşılaştırma
labels_sorted = sorted(auc_results.keys(), key=lambda x: auc_results[x], reverse=True)
ours_vals = [auc_results[l] for l in labels_sorted]
wang_vals = [WANG_AUC.get(l, 0) for l in labels_sorted]

y_pos = np.arange(len(labels_sorted))
axes[0].barh(y_pos + 0.15, wang_vals, 0.3, label='Wang et al. 2017', color='#bdc3c7', alpha=0.8)
axes[0].barh(y_pos - 0.15, ours_vals, 0.3, label='TorchXRayVision', color='#3498db')
# Nodule ve Mass'ı vurgula
for i, l in enumerate(labels_sorted):
    if l in ('Nodule', 'Mass'):
        axes[0].barh(i - 0.15, auc_results[l], 0.3, color='#e74c3c')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(labels_sorted)
axes[0].set_xlim(0.5, 1.0)
axes[0].set_title('AUC-ROC Karşılaştırma', fontsize=13, fontweight='bold')
axes[0].set_xlabel('AUC-ROC')
axes[0].legend(loc='lower right')
axes[0].axvline(x=0.7, color='gray', linestyle='--', alpha=0.3)
axes[0].axvline(x=0.8, color='gray', linestyle='--', alpha=0.3)
axes[0].invert_yaxis()

# 2. AlpCAN odak — Nodule & Mass ROC
for label in ('Nodule', 'Mass', 'Consolidation', 'Infiltration'):
    if label in LABEL_MAP:
        txrv_idx = TXRV_LABELS.index(LABEL_MAP[label])
        y_true = eval_processed[label].values
        y_score = predictions[:, txrv_idx]
        if y_true.sum() > 0:
            fpr, tpr, _ = roc_curve(y_true, y_score)
            auc = auc_results[label]
            lw = 3 if label in ('Nodule', 'Mass') else 1.5
            axes[1].plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})", linewidth=lw)

axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Eğrileri — AlpCAN Odak Patolojileri', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.2)

# 3. Precision-Recall eğrileri
for label in ('Nodule', 'Mass'):
    if label in LABEL_MAP:
        txrv_idx = TXRV_LABELS.index(LABEL_MAP[label])
        y_true = eval_processed[label].values
        y_score = predictions[:, txrv_idx]
        if y_true.sum() > 0:
            prec, rec, _ = precision_recall_curve(y_true, y_score)
            ap = ap_results[label]
            axes[2].plot(rec, prec, label=f"{label} (AP={ap:.3f})", linewidth=2)
            # Baseline (prevalence)
            prevalence = y_true.mean()
            axes[2].axhline(y=prevalence, color='gray', linestyle=':', alpha=0.3)

axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall — Nodule & Mass', fontsize=13, fontweight='bold')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: cxr_performance.png")

---
## 5. Threshold Analizi

In [ ]:
# Nodule ve Mass için detaylı threshold analizi
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

optimal_thresholds = {}

for idx, label in enumerate(['Nodule', 'Mass']):
    if label not in LABEL_MAP:
        continue

    txrv_idx = TXRV_LABELS.index(LABEL_MAP[label])
    y_true = eval_processed[label].values
    y_score = predictions[:, txrv_idx]

    # 1. Score dağılımı
    ax = axes[idx, 0]
    ax.hist(y_score[y_true == 0], bins=80, alpha=0.6, label=f'Negatif (n={sum(y_true==0):,})',
            color='#3498db', density=True)
    ax.hist(y_score[y_true == 1], bins=80, alpha=0.6, label=f'Pozitif (n={sum(y_true==1):,})',
            color='#e74c3c', density=True)
    ax.set_title(f'{label} — Tahmin Skoru Dağılımı', fontweight='bold')
    ax.set_xlabel('Skor')
    ax.set_ylabel('Yoğunluk')
    ax.legend()

    # 2. Sensitivity / Specificity vs Threshold
    ax = axes[idx, 1]
    thresholds = np.arange(0.0, 1.0, 0.005)
    sensitivities, specificities, ppvs, npvs = [], [], [], []

    for th in thresholds:
        y_pred = (y_score >= th).astype(int)
        tp = ((y_pred == 1) & (y_true == 1)).sum()
        tn = ((y_pred == 0) & (y_true == 0)).sum()
        fp = ((y_pred == 1) & (y_true == 0)).sum()
        fn = ((y_pred == 0) & (y_true == 1)).sum()

        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0

        sensitivities.append(sens)
        specificities.append(spec)
        ppvs.append(ppv)
        npvs.append(npv)

    ax.plot(thresholds, sensitivities, label='Sensitivity', color='#e74c3c', linewidth=2)
    ax.plot(thresholds, specificities, label='Specificity', color='#3498db', linewidth=2)
    ax.plot(thresholds, ppvs, label='PPV', color='#27ae60', linewidth=1, linestyle='--')
    ax.plot(thresholds, npvs, label='NPV', color='#8e44ad', linewidth=1, linestyle='--')

    # Optimal threshold (Youden's J)
    j_scores = np.array(sensitivities) + np.array(specificities) - 1
    opt_idx = np.argmax(j_scores)
    opt_th = thresholds[opt_idx]
    optimal_thresholds[label] = opt_th

    # Yüksek sensitivity threshold (tarama için ≥0.90 sensitivity hedef)
    high_sens_idx = None
    for i, s in enumerate(sensitivities):
        if s >= 0.90:
            high_sens_idx = i

    ax.axvline(x=opt_th, color='#f39c12', linestyle='--', alpha=0.8,
               label=f'Youden opt: {opt_th:.3f}')
    if high_sens_idx is not None:
        hs_th = thresholds[high_sens_idx]
        ax.axvline(x=hs_th, color='#e74c3c', linestyle=':', alpha=0.8,
                   label=f'Sens≥0.90: {hs_th:.3f}')

    ax.set_title(f'{label} — Threshold Analizi', fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # 3. F1 ve Matthews Correlation
    ax = axes[idx, 2]
    f1_scores = []
    for s, p in zip(sensitivities, ppvs):
        f1 = 2 * s * p / (s + p) if (s + p) > 0 else 0
        f1_scores.append(f1)

    ax.plot(thresholds, f1_scores, label='F1 Score', color='#2c3e50', linewidth=2)
    ax.plot(thresholds, j_scores, label="Youden's J", color='#e67e22', linewidth=2)
    f1_opt = thresholds[np.argmax(f1_scores)]
    ax.axvline(x=f1_opt, color='#2c3e50', linestyle='--', alpha=0.5,
               label=f'F1 opt: {f1_opt:.3f}')
    ax.set_title(f'{label} — F1 & Youden J', fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.legend()
    ax.grid(True, alpha=0.2)

    print(f"\n{label}:")
    print(f"  Youden optimal threshold: {opt_th:.3f}")
    print(f"    Sensitivity: {sensitivities[opt_idx]:.3f}, Specificity: {specificities[opt_idx]:.3f}")
    print(f"    PPV: {ppvs[opt_idx]:.3f}, NPV: {npvs[opt_idx]:.3f}")
    if high_sens_idx is not None:
        print(f"  High-sensitivity threshold (≥0.90): {thresholds[high_sens_idx]:.3f}")
        print(f"    Sensitivity: {sensitivities[high_sens_idx]:.3f}, Specificity: {specificities[high_sens_idx]:.3f}")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Saved: cxr_threshold_analysis.png")

---
## 6. AlpCAN CXR Ensemble Simülasyonu

In [ ]:
# AlpCAN CXR Pipeline — 4 model ensemble simülasyonu
# Gerçek modeller: Ark+, TorchXRayVision, X-Raydar, MedRAX
# Simülasyon: TorchXRayVision'ın farklı patoloji skorlarını farklı model proxy'si olarak kullan

ENSEMBLE_DECISIONS = {
    4: {"priority": "YÜKSEK ÖNCELİK", "action": "Acil BT önerisi — radyologa anlık uyarı", "color": "#e74c3c"},
    3: {"priority": "ORTA-YÜKSEK", "action": "2 hafta içinde BT önerilir", "color": "#e67e22"},
    2: {"priority": "ORTA", "action": "Radyolog değerlendirmesi istenir", "color": "#f39c12"},
    1: {"priority": "DÜŞÜK", "action": "Kayıt altına alınır, takip hatırlatıcı", "color": "#27ae60"},
    0: {"priority": "NORMAL", "action": "Rutin akış devam eder", "color": "#3498db"},
}

if 'Nodule' in LABEL_MAP and 'Mass' in LABEL_MAP:
    nodule_idx = TXRV_LABELS.index(LABEL_MAP['Nodule'])
    mass_idx = TXRV_LABELS.index(LABEL_MAP['Mass'])

    # Combined suspicion score
    combined_scores = np.maximum(predictions[:, nodule_idx], predictions[:, mass_idx])

    # 4 model simülasyonu — farklı hassasiyet seviyeleri
    model_configs = {
        'Ark+ (Swin-L)':         {'threshold': optimal_thresholds.get('Nodule', 0.35) * 0.85},
        'TorchXRayVision':       {'threshold': optimal_thresholds.get('Nodule', 0.40)},
        'X-Raydar (ResNet+Tr)':  {'threshold': optimal_thresholds.get('Nodule', 0.40) * 1.10},
        'MedRAX (MedSAM2)':      {'threshold': optimal_thresholds.get('Nodule', 0.35) * 0.80},
    }

    votes = np.zeros(len(combined_scores), dtype=int)
    print("Model Konfigürasyonları (Simüle):")
    for name, cfg in model_configs.items():
        model_votes = (combined_scores >= cfg['threshold']).astype(int)
        votes += model_votes
        print(f"  {name:>25s}: threshold={cfg['threshold']:.3f}, pozitif={model_votes.sum():,}")

    # Karar dağılımı
    print(f"\n{'=' * 65}")
    print(f"AlpCAN CXR Ensemble — Karar Dağılımı (N={len(votes):,})")
    print(f"{'=' * 65}")

    vote_counts = pd.Series(votes).value_counts().sort_index()
    for v in range(5):
        count = vote_counts.get(v, 0)
        pct = count / len(votes) * 100
        d = ENSEMBLE_DECISIONS[v]
        print(f"  {v}/4 oy → {d['priority']:>15s}: {count:>7,} ({pct:>5.1f}%) — {d['action']}")

    # Gerçek Nodule/Mass arasında
    true_positive_mask = (eval_processed['Nodule'].values == 1) | (eval_processed['Mass'].values == 1)
    true_negative_mask = eval_processed['No Finding'].values == 1

    print(f"\n  Gerçek Nodule/Mass ({true_positive_mask.sum():,}) oylama:")
    tp_votes = votes[true_positive_mask]
    for v in range(5):
        count = (tp_votes == v).sum()
        pct = count / len(tp_votes) * 100 if len(tp_votes) > 0 else 0
        print(f"    {v}/4 oy: {count:>5,} ({pct:>5.1f}%)")

    print(f"\n  Normal ({true_negative_mask.sum():,}) oylama:")
    tn_votes = votes[true_negative_mask]
    for v in range(5):
        count = (tn_votes == v).sum()
        pct = count / len(tn_votes) * 100 if len(tn_votes) > 0 else 0
        print(f"    {v}/4 oy: {count:>5,} ({pct:>5.1f}%)")

    # Sensitivity @ different ensemble thresholds
    print(f"\n  Ensemble karar eşiği analizi (Nodule/Mass tespit):")
    for min_votes in [1, 2, 3, 4]:
        ensemble_pred = (votes >= min_votes).astype(int)
        tp = (ensemble_pred[true_positive_mask] == 1).sum()
        fn = (ensemble_pred[true_positive_mask] == 0).sum()
        fp = (ensemble_pred[true_negative_mask] == 1).sum()
        tn = (ensemble_pred[true_negative_mask] == 0).sum()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        print(f"    ≥{min_votes}/4 oy: Sensitivity={sens:.3f}, Specificity={spec:.3f}")

In [ ]:
# Ensemble görselleştirmesi
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Oylama dağılımı — tüm veri
vote_vals = [vote_counts.get(v, 0) for v in range(5)]
vote_colors = [ENSEMBLE_DECISIONS[v]['color'] for v in range(5)]
vote_labels = [f"{v}/4 — {ENSEMBLE_DECISIONS[v]['priority']}" for v in range(5)]
axes[0].barh(vote_labels, vote_vals, color=vote_colors)
axes[0].set_title(f'Ensemble Oylama Dağılımı (N={len(votes):,})', fontweight='bold')
axes[0].set_xlabel('Görüntü Sayısı')
axes[0].invert_yaxis()

# 2. True Positive vs True Negative oylama karşılaştırma
tp_dist = [(tp_votes == v).sum() / len(tp_votes) * 100 if len(tp_votes) > 0 else 0 for v in range(5)]
tn_dist = [(tn_votes == v).sum() / len(tn_votes) * 100 if len(tn_votes) > 0 else 0 for v in range(5)]

x = np.arange(5)
axes[1].bar(x - 0.15, tp_dist, 0.3, label='Nodule/Mass (+)', color='#e74c3c')
axes[1].bar(x + 0.15, tn_dist, 0.3, label='Normal (-)', color='#3498db')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{v}/4' for v in range(5)])
axes[1].set_title('Oylama: Pozitif vs Normal', fontweight='bold')
axes[1].set_xlabel('Oy Sayısı')
axes[1].set_ylabel('Yüzde (%)')
axes[1].legend()

# 3. Pie chart — overall
axes[2].pie(vote_vals, labels=[f"{v}/4" for v in range(5)], colors=vote_colors,
            autopct='%1.1f%%', startangle=90)
axes[2].set_title('Genel Karar Dağılımı', fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cxr_ensemble_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: cxr_ensemble_analysis.png")

---
## 7. Örnek Tahminler ve BBox Doğrulama

In [ ]:
# En yüksek ve düşük skorlu örnekler
if 'Nodule' in LABEL_MAP:
    nodule_scores = predictions[:, TXRV_LABELS.index(LABEL_MAP['Nodule'])]

    # True Positive (yüksek skor + gerçek nodule)
    tp_mask = eval_processed['Nodule'].values == 1
    tp_indices = np.where(tp_mask)[0]
    tp_sorted = tp_indices[np.argsort(nodule_scores[tp_indices])[::-1]]

    # False Negative (düşük skor + gerçek nodule) — kaçırılanlar
    fn_sorted = tp_indices[np.argsort(nodule_scores[tp_indices])]

    # True Negative (normal + düşük skor)
    tn_mask = eval_processed['No Finding'].values == 1
    tn_indices = np.where(tn_mask)[0]

    # False Positive (yüksek skor + normal)
    fp_sorted = tn_indices[np.argsort(nodule_scores[tn_indices])[::-1]]

    fig, axes = plt.subplots(3, 4, figsize=(18, 14))

    # Row 1: True Positive
    for i, idx in enumerate(tp_sorted[:4]):
        img_name = all_image_names[idx]
        img_path = image_index.get(img_name)
        if img_path and img_path.exists():
            img = Image.open(img_path).convert('L')
            axes[0, i].imshow(img, cmap='gray')
            score = nodule_scores[idx]
            gt = eval_processed.iloc[idx]['Finding Labels'][:35]
            axes[0, i].set_title(f"Skor: {score:.3f}\n{gt}", fontsize=8, color='darkgreen')
        axes[0, i].axis('off')
    axes[0, 0].set_ylabel('True Positive\n(Doğru tespit)', fontsize=11, fontweight='bold', color='green')

    # Row 2: False Negative
    for i, idx in enumerate(fn_sorted[:4]):
        img_name = all_image_names[idx]
        img_path = image_index.get(img_name)
        if img_path and img_path.exists():
            img = Image.open(img_path).convert('L')
            axes[1, i].imshow(img, cmap='gray')
            score = nodule_scores[idx]
            gt = eval_processed.iloc[idx]['Finding Labels'][:35]
            axes[1, i].set_title(f"Skor: {score:.3f}\n{gt}", fontsize=8, color='darkred')
        axes[1, i].axis('off')
    axes[1, 0].set_ylabel('False Negative\n(Kaçırılan)', fontsize=11, fontweight='bold', color='red')

    # Row 3: False Positive
    for i, idx in enumerate(fp_sorted[:4]):
        img_name = all_image_names[idx]
        img_path = image_index.get(img_name)
        if img_path and img_path.exists():
            img = Image.open(img_path).convert('L')
            axes[2, i].imshow(img, cmap='gray')
            score = nodule_scores[idx]
            gt = eval_processed.iloc[idx]['Finding Labels'][:35]
            axes[2, i].set_title(f"Skor: {score:.3f}\n{gt}", fontsize=8, color='darkorange')
        axes[2, i].axis('off')
    axes[2, 0].set_ylabel('False Positive\n(Yanlış alarm)', fontsize=11, fontweight='bold', color='orange')

    plt.suptitle('TorchXRayVision Nodule Tahmin Örnekleri', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'cxr_prediction_examples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved: cxr_prediction_examples.png")

In [ ]:
# BBox ile lokalize doğrulama (varsa)
if bbox_df is not None:
    # Nodule/Mass BBox örnekleri
    bbox_nodule = bbox_df[bbox_df['Finding Label'].isin(['Nodule', 'Mass'])]
    print(f"BBox'lu Nodule/Mass görüntüleri: {bbox_nodule['Image Index'].nunique():,}")

    # BBox olan görüntülerde model performansı
    bbox_images = set(bbox_nodule['Image Index'].values)
    bbox_mask = eval_processed['Image Index'].isin(bbox_images)

    if bbox_mask.sum() > 0:
        bbox_scores = nodule_scores[bbox_mask]
        print(f"\nBBox'lu görüntülerde Nodule skoru:")
        print(f"  Mean: {bbox_scores.mean():.4f}")
        print(f"  Median: {np.median(bbox_scores):.4f}")
        print(f"  >0.5: {(bbox_scores > 0.5).sum()} / {len(bbox_scores)} ({(bbox_scores > 0.5).mean()*100:.1f}%)")

    # BBox overlay örneği (ilk 4)
    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    bbox_samples = bbox_nodule.groupby('Image Index').first().reset_index().head(4)

    for i, (_, row) in enumerate(bbox_samples.iterrows()):
        img_name = row['Image Index']
        img_path = image_index.get(img_name)
        if img_path and img_path.exists():
            img = Image.open(img_path).convert('L')
            axes[i].imshow(img, cmap='gray')

            # BBox çiz — orijinal koordinatları resme ölçekle
            img_w, img_h = img.size
            bboxes = bbox_nodule[bbox_nodule['Image Index'] == img_name]
            for _, b in bboxes.iterrows():
                x, y, w, h = b['Bbox [x'], b['y'], b['w'], b['h]']
                rect = plt.Rectangle((x, y), w, h,
                                     linewidth=2, edgecolor='red', facecolor='none')
                axes[i].add_patch(rect)

            # Model skoru
            img_mask = eval_processed['Image Index'] == img_name
            if img_mask.sum() > 0:
                idx_in_pred = np.where(img_mask)[0][0]
                score = nodule_scores[idx_in_pred]
                axes[i].set_title(f"Skor: {score:.3f}\n{row['Finding Label']}", fontsize=9)

        axes[i].axis('off')

    plt.suptitle('BBox Annotasyonlu Nodule/Mass Örnekleri + Model Skoru', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'cxr_bbox_examples.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Saved: cxr_bbox_examples.png")

---
## 8. Sonuçları Kaydet

In [ ]:
# Tüm tahminleri CSV
results_df = pd.DataFrame({'image_name': all_image_names})
for i, label in enumerate(TXRV_LABELS):
    results_df[f'pred_{label}'] = predictions[:, i]
for label in ALL_LABELS:
    gt_vals = []
    for img_name in all_image_names:
        row = eval_processed[eval_processed['Image Index'] == img_name]
        gt_vals.append(row[label].values[0] if len(row) > 0 else -1)
    results_df[f'gt_{label}'] = gt_vals

results_df.to_csv(OUTPUT_DIR / 'cxr_predictions_full.csv', index=False)
print(f"✅ cxr_predictions_full.csv ({len(results_df):,} rows, {results_df.shape[1]} cols)")

# AUC sonuçları
auc_df = pd.DataFrame([
    {
        'pathology': k,
        'auc_roc': v,
        'ap': ap_results.get(k, 0),
        'n_positive': int(eval_processed[k].sum()),
        'prevalence': eval_processed[k].mean(),
        'wang_2017_auc': WANG_AUC.get(k, None),
        'improvement': v - WANG_AUC.get(k, 0) if k in WANG_AUC else None,
        'optimal_threshold': optimal_thresholds.get(k, None),
    }
    for k, v in auc_results.items()
]).sort_values('auc_roc', ascending=False)

auc_df.to_csv(OUTPUT_DIR / 'cxr_auc_scores.csv', index=False)
print(f"✅ cxr_auc_scores.csv")

# Label dağılımı
dist_df = pd.DataFrame([
    {'label': l, 'count': int(eval_processed[l].sum()),
     'prevalence': eval_processed[l].mean()}
    for l in ALL_LABELS + ['No Finding']
]).sort_values('count', ascending=False)
dist_df.to_csv(OUTPUT_DIR / 'cxr_label_distribution.csv', index=False)
print(f"✅ cxr_label_distribution.csv")

In [ ]:
# Final özet
print("\n" + "=" * 70)
print("AlpCAN CXR Pipeline — TorchXRayVision Baseline ÖZET")
print("=" * 70)
print(f"\n📊 Dataset: NIH ChestX-ray14")
print(f"   Toplam görüntü: {len(eval_df):,}")
print(f"   Toplam hasta: {eval_df['Patient ID'].nunique():,}")
print(f"   İşlenen: {predictions.shape[0]:,}")
print(f"\n🤖 Model: TorchXRayVision DenseNet-121 (densenet121-res224-all)")
print(f"   18 patoloji tahmini, {n_params/1e6:.1f}M parametre")
if auc_results:
    print(f"\n📈 Performans:")
    print(f"   Ortalama AUC: {np.mean(list(auc_results.values())):.4f}")
    if 'Nodule' in auc_results:
        print(f"   Nodule AUC: {auc_results['Nodule']:.4f} (Wang: {WANG_AUC.get('Nodule', 'N/A')})")
    if 'Mass' in auc_results:
        print(f"   Mass AUC: {auc_results['Mass']:.4f} (Wang: {WANG_AUC.get('Mass', 'N/A')})")
print(f"\n📁 Çıktı dosyaları:")
for f in sorted(OUTPUT_DIR.glob('cxr_*')):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   {f.name} ({size_mb:.1f} MB)")
print(f"\n🎯 Sonraki adımlar:")
print(f"   1. Ark+ Foundation Model baseline (Agent 7A)")
print(f"   2. X-Raydar ResNet+Transformer (Agent 7C)")
print(f"   3. 4 model gerçek ensemble entegrasyonu")
print("=" * 70)